
# Seq2Seq 的數學推導

> 定義 Seq2Seq（vanilla RNN encoder–decoder）的 forward，再從 decoder output layer 開始做 BPTT（Backpropagation Through Time）完整推導。  
> 記號以向量形式表示，$\odot$ 表示逐元素相乘。  
> 本 notebook 對應的是 **encoder 與 decoder 都使用 vanilla RNN、decoder 初始 hidden 直接接 encoder 最後 hidden、training 使用 teacher forcing、loss 使用 average cross-entropy** 的版本。

Seq2Seq（Sequence-to-Sequence）是一種把一段序列轉換成另一段序列的模型。

可以把它想成：

* 輸入：一串東西（文字、數字、符號）
* 輸出：另一串東西（通常長度不一定一樣）

> 核心概念只有一句話：「先讀完整段輸入 → 再一個一個生成輸出」

<img src="image/Seq2seq_ani.gif" alt="seq2seq" style="width: 70%;"/>

---

## 0) 符號定義

給定：

- source sequence：
  $$
  \mathbf{x} = (\mathbf{x}_1, \mathbf{x}_2, \dots, \mathbf{x}_{T_x})
  $$
- target sequence：
  $$
  \mathbf{y}^{*} = (\mathbf{y}^{*}_1, \mathbf{y}^{*}_2, \dots, \mathbf{y}^{*}_{T_y})
  $$

其中：

- $\mathbf{x}_t \in \mathbb{R}^{V_x}$：encoder 在時間步 $t$ 的輸入（one-hot）
- $\mathbf{u}_t \in \mathbb{R}^{V_y}$：decoder 在時間步 $t$ 的輸入（teacher forcing 下通常為前一個正確 token 的 one-hot）
- $\mathbf{h}^{enc}_t \in \mathbb{R}^{H}$：encoder hidden state
- $\mathbf{h}^{dec}_t \in \mathbb{R}^{H}$：decoder hidden state
- $\mathbf{c} \in \mathbb{R}^{H}$：context vector
- $\mathbf{o}_t \in \mathbb{R}^{V_y}$：decoder 的 logits
- $\hat{\mathbf{y}}_t \in \mathbb{R}^{V_y}$：decoder 的 softmax 機率輸出

若採用 encoder / decoder 分開參數的寫法：

- encoder 參數：$(W_{xh}^{enc}, W_{hh}^{enc}, \mathbf{b}_h^{enc})$
- decoder 參數：$(W_{xh}^{dec}, W_{hh}^{dec}, \mathbf{b}_h^{dec})$
- output layer 參數：$(W_{ho}, \mathbf{b}_o)$

---

## 1) Seq2Seq Forward（向前傳播）



### 1.1 Encoder RNN

初始化：
$$
\mathbf{h}^{enc}_0 = \mathbf{0}
$$

在時間步 $t$，encoder 的 pre-activation 與 hidden state 為：
$$
\mathbf{a}^{enc}_t =
W_{xh}^{enc}\mathbf{x}_t + W_{hh}^{enc}\mathbf{h}^{enc}_{t-1} + \mathbf{b}_h^{enc}
$$

$$
\mathbf{h}^{enc}_t = \tanh(\mathbf{a}^{enc}_t)
$$

---

### 1.2 Context vector

encoder 讀完整個 source sequence 後，取最後 hidden state 作為 context vector：
$$
\mathbf{c} = \mathbf{h}^{enc}_{T_x}
$$

這也是 encoder 與 decoder 的橋接點。

---

### 1.3 Decoder RNN

decoder 初始 hidden state 設為：
$$
\mathbf{h}^{dec}_0 = \mathbf{c}
$$

若使用 teacher forcing，則 decoder 在時間步 $t$ 的輸入為：
$$
\mathbf{u}_t = \mathbf{y}^{*}_{t-1}
$$

其中 $t=1$ 時通常使用 `<sos>` 的 one-hot 向量。

decoder 的 pre-activation 與 hidden state：
$$
\mathbf{a}^{dec}_t =
W_{xh}^{dec}\mathbf{u}_t + W_{hh}^{dec}\mathbf{h}^{dec}_{t-1} + \mathbf{b}_h^{dec}
$$

$$
\mathbf{h}^{dec}_t = \tanh(\mathbf{a}^{dec}_t)
$$

---

### 1.4 Output layer

decoder 在每個時間步的 logits：
$$
\mathbf{o}_t = W_{ho}\mathbf{h}^{dec}_t + \mathbf{b}_o
$$

softmax 機率：
$$
\hat{\mathbf{y}}_t = \mathrm{softmax}(\mathbf{o}_t)
$$



## 2) Loss Function

若 target sequence 長度為 $T_y$，則每個時間步的 cross-entropy loss 為：
$$
\mathcal{L}_t = - \sum_{i=1}^{V_y} y^{*}_{t,i} \log \hat{y}_{t,i}
$$

整段 target sequence 的平均 loss 定義為：
$$
\mathcal{L} = \frac{1}{T_y}\sum_{t=1}^{T_y}\mathcal{L}_t
$$

這裡特別使用 **average loss**，因此 backward 時每個時間步的 logits 梯度也要除以 $T_y$。

---



## 3) 從 Output Layer 開始 Backward（Decoder 端）

### 3.1 Softmax + Cross Entropy 的梯度

定義：
$$
\delta_t^{o} \equiv \frac{\partial \mathcal{L}}{\partial \mathbf{o}_t}
$$

由 softmax 搭配 cross-entropy 的標準結果可得：
$$
\frac{\partial \mathcal{L}_t}{\partial \mathbf{o}_t}
=
\hat{\mathbf{y}}_t - \mathbf{y}^{*}_t
$$

但因為總 loss 是平均：
$$
\mathcal{L} = \frac{1}{T_y}\sum_{t=1}^{T_y}\mathcal{L}_t
$$

所以：
$$
\delta_t^{o}
=
\frac{\partial \mathcal{L}}{\partial \mathbf{o}_t}
=
\frac{1}{T_y}\left(\hat{\mathbf{y}}_t - \mathbf{y}^{*}_t\right)
$$

---

### 3.2 對 output layer 參數與 decoder hidden 的梯度

由
$$
\mathbf{o}_t = W_{ho}\mathbf{h}^{dec}_t + \mathbf{b}_o
$$

可得：

對 $W_{ho}$：
$$
\frac{\partial \mathcal{L}}{\partial W_{ho}}
=
\sum_{t=1}^{T_y}\delta_t^{o}(\mathbf{h}^{dec}_t)^T
$$

對 $\mathbf{b}_o$：
$$
\frac{\partial \mathcal{L}}{\partial \mathbf{b}_o}
=
\sum_{t=1}^{T_y}\delta_t^{o}
$$

對 $\mathbf{h}^{dec}_t$（來自當前 output layer）：
$$
\left.\frac{\partial \mathcal{L}}{\partial \mathbf{h}^{dec}_t}\right|_{\text{output}}
=
W_{ho}^T\delta_t^{o}
$$

---

### 3.3 Decoder hidden state 的總梯度

在 BPTT 中，$\mathbf{h}^{dec}_t$ 不只影響當前時間步輸出，也會透過 recurrent connection 影響未來時間步，因此定義：

$$
\delta_t^{h,dec} \equiv \frac{\partial \mathcal{L}}{\partial \mathbf{h}^{dec}_t}
$$

則有：
$$
\delta_t^{h,dec}
=
\underbrace{W_{ho}^T\delta_t^{o}}_{\text{當前輸出層}}
+
\underbrace{\delta_{t,\text{future}}^{h,dec}}_{\text{來自未來時間步}}
$$

在程式實作中，後面這項通常會寫成 `dh_next`。



## 4) Decoder RNN 的反向傳播（BPTT）

由 decoder hidden state 的定義：
$$
\mathbf{h}^{dec}_t = \tanh(\mathbf{a}^{dec}_t)
$$

其中
$$
\mathbf{a}^{dec}_t =
W_{xh}^{dec}\mathbf{u}_t + W_{hh}^{dec}\mathbf{h}^{dec}_{t-1} + \mathbf{b}_h^{dec}
$$

---

### 4.1 從 hidden 回傳到 pre-activation

因為
$$
\frac{d}{dz}\tanh(z)=1-\tanh^2(z)
$$

定義：
$$
\delta_t^{a,dec}
\equiv
\frac{\partial \mathcal{L}}{\partial \mathbf{a}^{dec}_t}
$$

則：
$$
\delta_t^{a,dec}
=
\delta_t^{h,dec}
\odot
\left(1-(\mathbf{h}^{dec}_t)^2\right)
$$

---

### 4.2 對 decoder 參數的梯度

對 $W_{xh}^{dec}$：
$$
\frac{\partial \mathcal{L}}{\partial W_{xh}^{dec}}
=
\sum_{t=1}^{T_y}\delta_t^{a,dec}\mathbf{u}_t^T
$$

對 $W_{hh}^{dec}$：
$$
\frac{\partial \mathcal{L}}{\partial W_{hh}^{dec}}
=
\sum_{t=1}^{T_y}\delta_t^{a,dec}(\mathbf{h}^{dec}_{t-1})^T
$$

對 $\mathbf{b}_h^{dec}$：
$$
\frac{\partial \mathcal{L}}{\partial \mathbf{b}_h^{dec}}
=
\sum_{t=1}^{T_y}\delta_t^{a,dec}
$$

---

### 4.3 對前一個 decoder hidden 的梯度

由
$$
\mathbf{a}^{dec}_t =
W_{xh}^{dec}\mathbf{u}_t + W_{hh}^{dec}\mathbf{h}^{dec}_{t-1} + \mathbf{b}_h^{dec}
$$

可得：
$$
\delta_{t-1}^{h,dec}
=
(W_{hh}^{dec})^T \delta_t^{a,dec}
$$

這一項就是下一輪 backward 迴圈中的 future hidden gradient。



## 5) Encoder–Decoder 的梯度橋接（Seq2Seq 最關鍵）

這個 vanilla Seq2Seq 的橋接關係是：
$$
\mathbf{h}^{dec}_0 = \mathbf{c} = \mathbf{h}^{enc}_{T_x}
$$

因此，當 decoder 做完 BPTT 之後，最前面那一包 hidden gradient 會直接變成 encoder 最後 hidden state 的梯度：

$$
\frac{\partial \mathcal{L}}{\partial \mathbf{h}^{enc}_{T_x}}
=
\frac{\partial \mathcal{L}}{\partial \mathbf{h}^{dec}_0}
$$

若沿用前面的符號，可記為：
$$
\delta_{T_x}^{h,enc} = \delta_0^{h,dec}
$$

這條式子就是：

- decoder loss 為什麼能訓練 encoder
- 為什麼即使 encoder 自己沒有 output layer，仍然能收到梯度

的根本原因。

---



## 6) Encoder RNN 的反向傳播（BPTT）

encoder 的 hidden state：
$$
\mathbf{h}^{enc}_t = \tanh(\mathbf{a}^{enc}_t)
$$

其中
$$
\mathbf{a}^{enc}_t =
W_{xh}^{enc}\mathbf{x}_t + W_{hh}^{enc}\mathbf{h}^{enc}_{t-1} + \mathbf{b}_h^{enc}
$$

---

### 6.1 從最後 hidden 開始 backward

Encoder backward 的起點不是 output layer，而是 decoder 傳回來的 bridge gradient：
$$
\delta_{T_x}^{h,enc}
=
\frac{\partial \mathcal{L}}{\partial \mathbf{h}^{enc}_{T_x}}
$$

---

### 6.2 從 hidden 回傳到 pre-activation

定義：
$$
\delta_t^{a,enc}
\equiv
\frac{\partial \mathcal{L}}{\partial \mathbf{a}^{enc}_t}
$$

則：
$$
\delta_t^{a,enc}
=
\delta_t^{h,enc}
\odot
\left(1-(\mathbf{h}^{enc}_t)^2\right)
$$

---

### 6.3 對 encoder 參數的梯度

對 $W_{xh}^{enc}$：
$$
\frac{\partial \mathcal{L}}{\partial W_{xh}^{enc}}
=
\sum_{t=1}^{T_x}\delta_t^{a,enc}\mathbf{x}_t^T
$$

對 $W_{hh}^{enc}$：
$$
\frac{\partial \mathcal{L}}{\partial W_{hh}^{enc}}
=
\sum_{t=1}^{T_x}\delta_t^{a,enc}(\mathbf{h}^{enc}_{t-1})^T
$$

對 $\mathbf{b}_h^{enc}$：
$$
\frac{\partial \mathcal{L}}{\partial \mathbf{b}_h^{enc}}
=
\sum_{t=1}^{T_x}\delta_t^{a,enc}
$$

---

### 6.4 對前一個 encoder hidden 的梯度

$$
\delta_{t-1}^{h,enc}
=
(W_{hh}^{enc})^T \delta_t^{a,enc}
$$

因此 encoder 也和一般 RNN 一樣，從最後一步一路反傳到最前面。



## 7) 把整個梯度流串起來看

若把整個 Seq2Seq 畫成梯度流，會是：

$$
\mathcal{L}
\rightarrow
\mathbf{o}_{T_y}, \mathbf{o}_{T_y-1}, \dots, \mathbf{o}_1
\rightarrow
\mathbf{h}^{dec}_{T_y}, \mathbf{h}^{dec}_{T_y-1}, \dots, \mathbf{h}^{dec}_0
=
\mathbf{h}^{enc}_{T_x}
\rightarrow
\mathbf{h}^{enc}_{T_x-1}, \dots, \mathbf{h}^{enc}_1
$$

也就是：

1. 先從 decoder 的每一步 logits 反傳
2. 經過 output layer 回到 decoder hidden states
3. decoder 做完整 BPTT
4. 最前面的 decoder hidden gradient 接到 encoder 最後 hidden
5. encoder 再做一遍 BPTT

所以這個模型雖然看起來像「兩個 RNN」，但在訓練時其實是一張完整連接的計算圖。

---



## 8) 最後整理：單一時間步的 backward 公式

### 8.1 Decoder output layer
$$
\delta_t^{o}
=
\frac{1}{T_y}\left(\hat{\mathbf{y}}_t - \mathbf{y}^{*}_t\right)
$$

$$
\frac{\partial \mathcal{L}}{\partial W_{ho}}
=
\sum_{t=1}^{T_y}\delta_t^{o}(\mathbf{h}^{dec}_t)^T
$$

$$
\frac{\partial \mathcal{L}}{\partial \mathbf{b}_o}
=
\sum_{t=1}^{T_y}\delta_t^{o}
$$

$$
\delta_t^{h,dec}
=
W_{ho}^T\delta_t^{o}
+
\delta_{t,\text{future}}^{h,dec}
$$

---

### 8.2 Decoder recurrent layer
$$
\delta_t^{a,dec}
=
\delta_t^{h,dec}\odot \left(1-(\mathbf{h}^{dec}_t)^2\right)
$$

$$
\frac{\partial \mathcal{L}}{\partial W_{xh}^{dec}}
=
\sum_{t=1}^{T_y}\delta_t^{a,dec}\mathbf{u}_t^T
$$

$$
\frac{\partial \mathcal{L}}{\partial W_{hh}^{dec}}
=
\sum_{t=1}^{T_y}\delta_t^{a,dec}(\mathbf{h}^{dec}_{t-1})^T
$$

$$
\frac{\partial \mathcal{L}}{\partial \mathbf{b}_h^{dec}}
=
\sum_{t=1}^{T_y}\delta_t^{a,dec}
$$

$$
\delta_{t-1}^{h,dec}
=
(W_{hh}^{dec})^T\delta_t^{a,dec}
$$

---

### 8.3 Bridge
$$
\delta_{T_x}^{h,enc} = \delta_0^{h,dec}
$$

---

### 8.4 Encoder recurrent layer
$$
\delta_t^{a,enc}
=
\delta_t^{h,enc}\odot \left(1-(\mathbf{h}^{enc}_t)^2\right)
$$

$$
\frac{\partial \mathcal{L}}{\partial W_{xh}^{enc}}
=
\sum_{t=1}^{T_x}\delta_t^{a,enc}\mathbf{x}_t^T
$$

$$
\frac{\partial \mathcal{L}}{\partial W_{hh}^{enc}}
=
\sum_{t=1}^{T_x}\delta_t^{a,enc}(\mathbf{h}^{enc}_{t-1})^T
$$

$$
\frac{\partial \mathcal{L}}{\partial \mathbf{b}_h^{enc}}
=
\sum_{t=1}^{T_x}\delta_t^{a,enc}
$$

$$
\delta_{t-1}^{h,enc}
=
(W_{hh}^{enc})^T\delta_t^{a,enc}
$$



## 9) 對照程式實作的對應關係

若對照 NumPy 版 Seq2Seq 實作，會有以下對應：

- `dlogits /= T_dec`
  - 對應：
  $$
  \delta_t^{o}
  =
  \frac{1}{T_y}\left(\hat{\mathbf{y}}_t-\mathbf{y}^{*}_t\right)
  $$

- `self.dWho += np.outer(dlogits, h)`
  - 對應：
  $$
  \frac{\partial \mathcal{L}}{\partial W_{ho}}
  +=
  \delta_t^{o}(\mathbf{h}_t^{dec})^T
  $$

- `dh = self.Who.T @ dlogits + dh_next`
  - 對應：
  $$
  \delta_t^{h,dec}
  =
  W_{ho}^T\delta_t^{o}
  +
  \delta_{t,\text{future}}^{h,dec}
  $$

- `da = (1.0 - h * h) * dh`
  - 對應：
  $$
  \delta_t^{a,dec}
  =
  \delta_t^{h,dec}\odot(1-(\mathbf{h}^{dec}_t)^2)
  $$

- `self.dWxh_dec += np.outer(da, x)`
  - 對應：
  $$
  \frac{\partial \mathcal{L}}{\partial W_{xh}^{dec}}
  +=
  \delta_t^{a,dec}\mathbf{u}_t^T
  $$

- `self.dWhh_dec += np.outer(da, h_prev)`
  - 對應：
  $$
  \frac{\partial \mathcal{L}}{\partial W_{hh}^{dec}}
  +=
  \delta_t^{a,dec}(\mathbf{h}^{dec}_{t-1})^T
  $$

- `dh_next = self.Whh_dec.T @ da`
  - 對應：
  $$
  \delta_{t-1}^{h,dec}
  =
  (W_{hh}^{dec})^T\delta_t^{a,dec}
  $$

- `d_enc_last = dh_next.copy()`
  - 對應：
  $$
  \delta_{T_x}^{h,enc}=\delta_0^{h,dec}
  $$

- `da = (1.0 - h * h) * dh_next`
  - 對應：
  $$
  \delta_t^{a,enc}
  =
  \delta_t^{h,enc}\odot(1-(\mathbf{h}^{enc}_t)^2)
  $$

- `dh_next = self.Whh_enc.T @ da`
  - 對應：
  $$
  \delta_{t-1}^{h,enc}
  =
  (W_{hh}^{enc})^T\delta_t^{a,enc}
  $$

---



## 10) 一句話總結

這份 Seq2Seq（vanilla RNN）在數學上的核心是：

$$
\text{Encoder RNN}
\rightarrow
\mathbf{c}
\rightarrow
\text{Decoder RNN}
\rightarrow
\mathcal{L}
$$

而 backward 的核心則是：

$$
\mathcal{L}
\rightarrow
\text{Decoder BPTT}
\rightarrow
\mathbf{h}^{dec}_0
=
\mathbf{h}^{enc}_{T_x}
\rightarrow
\text{Encoder BPTT}
$$

也就是說：

> **loss 雖然定義在 decoder 端，但梯度會穿過 context vector，回傳整個 encoder。**

這正是最經典 Seq2Seq 訓練機制的本質；而後來加入 attention，就是為了改善所有 source 資訊都被壓進單一 context vector 的瓶頸。


# Seq2Seq Forward Pass（數值範例完整推導）

---

# 1. 問題設定

我們考慮一個簡單的 Seq2Seq 任務：

- 輸入（source）：A B  
- 輸出（target）：X Y  

---

## Vocabulary（one-hot）

### Source vocab

| token | 向量 |
|------|------|
| A | [1, 0, 0] |
| B | [0, 1, 0] |
| pad | [0, 0, 1] |

### Target vocab

| token | 向量 |
|------|------|
| sos | [1, 0, 0] |
| X | [0, 1, 0] |
| Y | [0, 0, 1] |

---

## 序列表示

$$
\mathbf{x}_1 = [1,0,0], \quad \mathbf{x}_2 = [0,1,0]
$$

$$
\mathbf{y}_1^* = X,\quad \mathbf{y}_2^* = Y
$$

Teacher forcing：

$$
\mathbf{u}_1 = <sos> = [1,0,0], \quad \mathbf{u}_2 = X = [0,1,0]
$$

---

# 2. 模型參數

## Encoder

$$
W_{xh}^{enc} =
\begin{bmatrix}
0.5 & -0.3 & 0.1 \\
0.2 & 0.4 & -0.5
\end{bmatrix}
$$

$$
W_{hh}^{enc} =
\begin{bmatrix}
0.1 & 0.2 \\
-0.3 & 0.4
\end{bmatrix}
$$

$$
b_h^{enc} = [0, 0]
$$

---

## Decoder

$$
W_{xh}^{dec} =
\begin{bmatrix}
0.3 & -0.2 & 0.1 \\
0.6 & 0.1 & -0.4
\end{bmatrix}
$$

$$
W_{hh}^{dec} =
\begin{bmatrix}
0.2 & 0.1 \\
-0.5 & 0.3
\end{bmatrix}
$$

$$
b_h^{dec} = [0,0]
$$

---

## Output layer

$$
W_{ho} =
\begin{bmatrix}
0.2 & -0.1 \\
0.5 & 0.3 \\
-0.4 & 0.2
\end{bmatrix}
$$

$$
b_o = [0,0,0]
$$

---

# 3. Encoder Forward

初始化：

$$
\mathbf{h}_0^{enc} = [0,0]
$$

---

## Step 1（輸入 A）

$$
\mathbf{x}_1 = [1,0,0]
$$

$$
\mathbf{a}_1^{enc} = W_{xh}^{enc}\mathbf{x}_1 =
\begin{bmatrix}
0.5 \\
0.2
\end{bmatrix}
$$

$$
\mathbf{h}_1^{enc} = \tanh([0.5,0.2])
$$

$$
\tanh(0.5) \approx 0.462,\quad \tanh(0.2) \approx 0.197
$$

$$
\mathbf{h}_1^{enc} \approx [0.462, 0.197]
$$

---

## Step 2（輸入 B）

$$
\mathbf{x}_2 = [0,1,0]
$$

$$
W_{xh}^{enc}\mathbf{x}_2 =
\begin{bmatrix}
-0.3 \\
0.4
\end{bmatrix}
$$

$$
W_{hh}^{enc}\mathbf{h}_1^{enc} =
\begin{bmatrix}
0.1 & 0.2 \\
-0.3 & 0.4
\end{bmatrix}
\begin{bmatrix}
0.462 \\
0.197
\end{bmatrix}
$$

逐項計算：

$$
=
\begin{bmatrix}
0.1 \cdot 0.462 + 0.2 \cdot 0.197 \\
-0.3 \cdot 0.462 + 0.4 \cdot 0.197
\end{bmatrix}
$$

$$
=
\begin{bmatrix}
0.0462 + 0.0394 \\
-0.1386 + 0.0788
\end{bmatrix}
=
\begin{bmatrix}
0.0856 \\
-0.0598
\end{bmatrix}
$$

$$
\mathbf{a}_2^{enc} =
[-0.3, 0.4] + [0.0856, -0.0598]
=
[-0.2144, 0.3402]
$$

$$
\mathbf{h}_2^{enc} = \tanh([-0.2144, 0.3402])
$$

$$
\tanh(-0.2144)\approx -0.211,\quad \tanh(0.3402)\approx 0.327
$$

$$
\mathbf{h}_2^{enc} \approx [-0.211, 0.327]
$$

---

## Context vector

$$
\mathbf{c} = \mathbf{h}_2^{enc} = [-0.211, 0.327]
$$

---

# 4. Decoder Forward

初始化：

$$
\mathbf{h}_0^{dec} = \mathbf{c}
$$

---

## Step 1（輸入 <sos>）

$$
\mathbf{u}_1 = [1,0,0]
$$

$$
W_{xh}^{dec}\mathbf{u}_1 =
\begin{bmatrix}
0.3 \\
0.6
\end{bmatrix}
$$

$$
W_{hh}^{dec}\mathbf{h}_0^{dec} =
\begin{bmatrix}
0.2 & 0.1 \\
-0.5 & 0.3
\end{bmatrix}
\begin{bmatrix}
-0.211 \\
0.327
\end{bmatrix}
$$

$$
=
\begin{bmatrix}
0.2(-0.211) + 0.1(0.327) \\
-0.5(-0.211) + 0.3(0.327)
\end{bmatrix}
$$

$$
=
\begin{bmatrix}
-0.0422 + 0.0327 \\
0.1055 + 0.0981
\end{bmatrix}
=
[-0.0095, 0.2036]
$$

$$
\mathbf{a}_1^{dec} =
[0.3,0.6] + [-0.0095,0.2036]
=
[0.2905, 0.8036]
$$

$$
\mathbf{h}_1^{dec} = \tanh([0.2905,0.8036])
$$

$$
\approx [0.283, 0.666]
$$

---

## Output Step 1

$$
\mathbf{o}_1 = W_{ho}\mathbf{h}_1^{dec}
$$

$$
=
\begin{bmatrix}
0.2 & -0.1 \\
0.5 & 0.3 \\
-0.4 & 0.2
\end{bmatrix}
\begin{bmatrix}
0.283 \\
0.666
\end{bmatrix}
$$

$$
=
\begin{bmatrix}
0.2(0.283) - 0.1(0.666) \\
0.5(0.283) + 0.3(0.666) \\
-0.4(0.283) + 0.2(0.666)
\end{bmatrix}
$$

$$
=
\begin{bmatrix}
0.0566 - 0.0666 \\
0.1415 + 0.1998 \\
-0.1132 + 0.1332
\end{bmatrix}
=
[-0.010, 0.341, 0.020]
$$

softmax：

$$
\hat{\mathbf{y}}_1 \approx [0.29, 0.39, 0.32]
$$

---

## Step 2（輸入 X）

$$
\mathbf{u}_2 = [0,1,0]
$$

$$
W_{xh}^{dec}\mathbf{u}_2 = [-0.2, 0.1]
$$

$$
W_{hh}^{dec}\mathbf{h}_1^{dec}
=
\begin{bmatrix}
0.2 & 0.1 \\
-0.5 & 0.3
\end{bmatrix}
\begin{bmatrix}
0.283 \\
0.666
\end{bmatrix}
$$

$$
=
\begin{bmatrix}
0.0566 + 0.0666 \\
-0.1415 + 0.1998
\end{bmatrix}
=
[0.1232, 0.0583]
$$

$$
\mathbf{a}_2^{dec} =
[-0.2,0.1] + [0.1232,0.0583]
=
[-0.0768, 0.1583]
$$

$$
\mathbf{h}_2^{dec} = \tanh([-0.0768,0.1583])
\approx [-0.077, 0.157]
$$

---

## Output Step 2

$$
\mathbf{o}_2 = W_{ho}\mathbf{h}_2^{dec}
\approx [-0.030, 0.001, 0.061]
$$

softmax：

$$
\hat{\mathbf{y}}_2 \approx [0.31, 0.32, 0.37]
$$

---

# 5. Forward 流程總結

Encoder：

$$
(A,B)
\rightarrow \mathbf{h}_1^{enc}
\rightarrow \mathbf{h}_2^{enc}
\rightarrow \mathbf{c}
$$

Decoder：

$$
\mathbf{c}
\rightarrow \mathbf{h}_1^{dec}
\rightarrow \hat{y}_1
\rightarrow \mathbf{h}_2^{dec}
\rightarrow \hat{y}_2
$$

---

# 6. 核心理解

$$
\text{Seq2Seq} =
\text{Encoder} \rightarrow \mathbf{c} \rightarrow \text{Decoder}
$$

> 將整個輸入序列壓縮成一個向量，再逐步生成輸出序列

# Seq2Seq Backward Propagation（數值範例）

## 📌 1. Loss 定義

$$
\mathcal{L} = \frac{1}{2} (\mathcal{L}_1 + \mathcal{L}_2)
$$

$$
\mathcal{L}_t = -\log \hat{y}_t[y_t^*]
$$

---

## 📌 2. Softmax + Cross Entropy 梯度

### Step 1（target = X = [0,1,0]）

預測：
$$
\hat{\mathbf{y}}_1 \approx [0.29, 0.39, 0.32]
$$

$$
\delta_1^o = \frac{1}{2}(\hat{\mathbf{y}}_1 - \mathbf{y}_1^*)
$$

$$
= \frac{1}{2}[0.29, 0.39-1, 0.32]
= [0.145, -0.305, 0.16]
$$

---

### Step 2（target = Y = [0,0,1]）

$$
\hat{\mathbf{y}}_2 \approx [0.31, 0.32, 0.37]
$$

$$
\delta_2^o = \frac{1}{2}(\hat{\mathbf{y}}_2 - \mathbf{y}_2^*)
$$

$$
= [0.155, 0.16, -0.315]
$$

---

## 📌 3. Output Layer 梯度

### Step 2

$$
\mathbf{h}_2^{dec} \approx [-0.077, 0.157]
$$

$$
\frac{\partial \mathcal{L}}{\partial W_{ho}}^{(t=2)} =
\delta_2^o (\mathbf{h}_2^{dec})^T
$$

$$
=
\begin{bmatrix}
0.155 \\
0.16 \\
-0.315
\end{bmatrix}
\begin{bmatrix}
-0.077 & 0.157
\end{bmatrix}
$$

$$
=
\begin{bmatrix}
-0.0119 & 0.0243 \\
-0.0123 & 0.0251 \\
0.0243 & -0.0495
\end{bmatrix}
$$

---

### Step 1

$$
\mathbf{h}_1^{dec} \approx [0.283, 0.666]
$$

$$
\frac{\partial \mathcal{L}}{\partial W_{ho}}^{(t=1)} =
\delta_1^o (\mathbf{h}_1^{dec})^T
$$

$$
=
\begin{bmatrix}
0.145 \\
-0.305 \\
0.16
\end{bmatrix}
\begin{bmatrix}
0.283 & 0.666
\end{bmatrix}
$$

$$
=
\begin{bmatrix}
0.041 & 0.097 \\
-0.086 & -0.203 \\
0.045 & 0.107
\end{bmatrix}
$$

---

### Total

$$
\frac{\partial \mathcal{L}}{\partial W_{ho}} =
\text{Step1} + \text{Step2}
$$

---

## 📌 4. 傳回 Decoder Hidden

### Step 2

$$
\delta_2^{h} = W_{ho}^T \delta_2^o
$$

$$
W_{ho}^T =
\begin{bmatrix}
0.2 & 0.5 & -0.4 \\
-0.1 & 0.3 & 0.2
\end{bmatrix}
$$

$$
\delta_2^h =
\begin{bmatrix}
0.2 & 0.5 & -0.4 \\
-0.1 & 0.3 & 0.2
\end{bmatrix}
\begin{bmatrix}
0.155 \\
0.16 \\
-0.315
\end{bmatrix}
$$

$$
=
[0.237, -0.0315]
$$

---

## 📌 5. tanh Backward

$$
\delta_2^{a} = \delta_2^h \odot (1 - h_2^2)
$$

$$
h_2^{dec} \approx [-0.077, 0.157]
$$

$$
1 - h^2 \approx [0.994, 0.975]
$$

$$
\delta_2^{a} \approx [0.235, -0.0307]
$$

---

## 📌 6. 傳回 Step 1

$$
\delta_1^{h, future} = W_{hh}^{dec T} \delta_2^{a}
$$

$$
W_{hh}^{dec T} =
\begin{bmatrix}
0.2 & -0.5 \\
0.1 & 0.3
\end{bmatrix}
$$

$$
\delta_1^{h, future} =
\begin{bmatrix}
0.0624 \\
0.0143
\end{bmatrix}
$$

---

## 📌 7. Step 1 Hidden 梯度

$$
\delta_1^h = W_{ho}^T \delta_1^o + \delta_1^{h,future}
$$

（略去中間矩陣乘法展開）

---

## 📌 8. Decoder → Encoder 梯度橋接

$$
\delta_{enc}^{h_{T_x}} = \delta_0^{h,dec}
$$

也就是：

$$
\delta_2^{enc} = \delta_1^{h,dec \, (最終傳回)}
$$

---

## 📌 9. Encoder Backward

### Step 2

$$
\delta_2^{a,enc} =
\delta_2^{h,enc} \odot (1 - (h_2^{enc})^2)
$$

$$
h_2^{enc} \approx [-0.211, 0.327]
$$

$$
1 - h^2 \approx [0.955, 0.893]
$$

---

### Step 1

$$
\delta_1^{h,enc} =
W_{hh}^{enc T} \delta_2^{a,enc}
$$

---

## 📌 10. 梯度流總結

$$
\mathcal{L}
\rightarrow \mathbf{o}_2 \rightarrow \mathbf{h}_2^{dec}
\rightarrow \mathbf{h}_1^{dec}
\rightarrow \mathbf{h}_0^{dec}
$$

$$
\mathbf{h}_0^{dec} = \mathbf{h}_2^{enc}
$$

$$
\rightarrow \mathbf{h}_1^{enc} \rightarrow \mathbf{h}_0^{enc}
$$

---

## 📌 11. 一句話總結

$$
\text{loss 只在 decoder，但梯度會完整流回 encoder}
$$

這就是 Seq2Seq 訓練的本質